In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd

df = pd.read_csv(path + "/" + "Q3_data.csv")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Columns like D_136 will be removed because the hold too many null values. other columns will be filled with mode or median accordingly
# Analyze missing values
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

df = df.drop(missing_data[(missing_data['Missing_Percentage'] > 35)]['Column'], axis=1) # Drop columns with more than 35% of the data missing

missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data

# Do we have categorical columns? if yes fill with the mode
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

numrical_columns = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")

print("Numerical Columns:", list(numrical_columns))

df[numrical_columns] = df[numrical_columns].fillna(df[numrical_columns].median())

print("No missing data left")

In [ ]:
# Task 2: Write your code here:

# Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
# No categorical. No need for encoding!

# Do we have categorical columns?
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

print("No categorical. No need for encoding!")

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

# Replace 'status' with target
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt
import seaborn as sns

# Target is imbalance, we need a stratified split!

def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Target")

print("Target is imbalance, we need a stratified split!")

In [ ]:
# Task 1: Write your code here:

X = df.drop("Target", axis=1).astype(float) # don't include the target in X
y = df['Target'].astype(float) # take only the target for y

In [ ]:
%pip install catboost -q

In [ ]:
# Task 2,3,4,5: Write your code here:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
model = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )

accuracy = []
f1 = []

# Binary

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  print(f"Training...")
  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

  # 3. Save metrics for that model in this fold
  accuracy.append(accuracy_score(y_test, y_pred))
  f1.append(f1_score(y_test, y_pred, zero_division=0))


print(f"Average F1-Score: {np.mean(f1)}") # F1 Score is the appropriate metric for this data since the target is imbalanced
print(f"Average Accuracy: {np.mean(accuracy)}")
print("F1 Score is the appropriate metric for this data since the target is imbalanced")

In [ ]:
# Task 1: Write your code here:
imp = model.feature_importances_

# Create a 1x3 plot
fig = plt.figure(figsize=(18, 6))
features = X.columns

model_name = "Random Forest"
  # Sort features by importance for a cleaner plot
sorted_idx = np.argsort(imp)

plt.barh(features[sorted_idx], imp[sorted_idx])
plt.title(f"{model_name} Feature Importance")
plt.xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:


absolute_coef = np.abs(model.feature_importances_)
sorted_idx = np.argsort(absolute_coef)


golden_feature = X.columns[sorted_idx[-1]]

print(f"The Golden Feature is {golden_feature}")

In [ ]:
# Task Bonus: Write your code here:
golden_X = X[[golden_feature]]
golden_X

In [ ]:
n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
model = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )

accuracy = []
f1 = []

# Binary

for fold_idx, (train_index, test_index) in enumerate(skf.split(golden_X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  print(f"Training...")
  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

  # 3. Save metrics for that model in this fold
  accuracy.append(accuracy_score(y_test, y_pred))
  f1.append(f1_score(y_test, y_pred, zero_division=0))


print(f"Average F1-Score: {np.mean(f1)}") # F1 Score is the appropriate metric for this data since the target is imbalanced
print(f"Average Accuracy: {np.mean(accuracy)}")
print("F1 Score is the appropriate metric for this data since the target is imbalanced")

In [ ]:
# The score is identical to the one with the features
# Scores including the other features:
# Average F1-Score: 0.6839714785086035
# Average Accuracy: 0.8366082854286429